# Reranker 消融實驗（Colab GPU）

**核心問題**：reranker 在 RT 上幫倒忙，是因為：
- A) RT 排序已夠好，reranker 反而打亂？
- B) 給 reranker 太複雜的輸入，簡單輸入反而有效？

**實驗設計**（6 管道 × 3 rerank depth）：
1. Dense only → reranker
2. BM25 only → reranker
3. RRF → reranker
4. RT → reranker（之前測過，幫倒忙）
5. Dense → reranker → 再跟 BM25 做 RT 融合（新管道！）
6. 各單獨組件 baseline（無 reranker）

**Reranker**: `cross-encoder/ms-marco-MiniLM-L-6-v2`（CPU 可跑）

**Dataset**: SciFact（快速，有快取）

In [1]:
!pip install -q beir sentence-transformers rank-bm25 numpy pytrec-eval-terrier
!nvidia-smi | head -5 || echo 'No GPU (CPU mode)'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 17.8 MB/s eta 0:00:00
Sat Mar 28 01:15:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |


In [2]:
import json
import os
import re
import time
from collections import defaultdict

import numpy as np
import pytrec_eval
import torch
from beir import util
from beir.datasets.data_loader import GenericDataLoader
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

/usr/local/lib/python3.12/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


Device: cuda


In [3]:
# ── Official evaluation (pytrec_eval) ──

def evaluate_official(qrels, run_dict, metrics=None):
    """Official BEIR evaluation with pytrec_eval."""
    if metrics is None:
        metrics = {"ndcg_cut_10", "recall_100", "map"}
    qrels_int = {
        qid: {did: int(rel) for did, rel in rels.items()}
        for qid, rels in qrels.items()
    }
    evaluator = pytrec_eval.RelevanceEvaluator(qrels_int, metrics)
    scores = evaluator.evaluate(run_dict)
    result = {}
    for metric in metrics:
        vals = [scores[qid].get(metric, 0) for qid in scores]
        result[metric] = round(sum(vals) / len(vals), 6)
    return result


# ── Fusion functions ──

def simple_rrf(b, d, k=5, bw=1.0, dw=1.2):
    scores = defaultdict(float)
    for rank, (did, _) in enumerate(b):
        scores[did] += bw / (k + rank + 1)
    for rank, (did, _) in enumerate(d):
        scores[did] += dw / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def riverbed_tension(
    b, d, k_low=3, k_high=10, top_n=20,
    boost_max=1.2, score_w=0.5, bw=0.8, dw=1.4,
):
    b_set = {did for did, _ in b[:top_n]}
    d_set = {did for did, _ in d[:top_n]}
    union = b_set | d_set
    agreement = len(b_set & d_set) / len(union) if union else 0.0
    tension = 1.0 - agreement
    adaptive_k = max(1, int(k_low + (k_high - k_low) * tension))
    boost = 1.0 + (boost_max - 1.0) * agreement

    rrf_scores = defaultdict(float)
    presence = defaultdict(int)
    for rank, (did, _) in enumerate(b):
        rrf_scores[did] += bw / (adaptive_k + rank + 1)
        presence[did] += 1
    for rank, (did, _) in enumerate(d):
        rrf_scores[did] += dw / (adaptive_k + rank + 1)
        presence[did] += 1
    for did in rrf_scores:
        if presence[did] >= 2:
            rrf_scores[did] *= boost

    def norm(results):
        if not results:
            return {}
        vals = [s for _, s in results]
        mn, mx = min(vals), max(vals)
        rng = mx - mn if mx > mn else 1.0
        return {did: (s - mn) / rng for did, s in results}

    b_n, d_n = norm(b), norm(d)
    rv = list(rrf_scores.values())
    r_mn, r_mx = min(rv), max(rv)
    r_rng = r_mx - r_mn if r_mx > r_mn else 1.0
    tw = bw + dw

    all_docs = set(rrf_scores) | set(b_n) | set(d_n)
    final = {}
    for did in all_docs:
        r = (rrf_scores.get(did, 0) - r_mn) / r_rng
        s = (bw * b_n.get(did, 0) + dw * d_n.get(did, 0)) / tw
        final[did] = (1 - score_w) * r + score_w * s
    return sorted(final.items(), key=lambda x: x[1], reverse=True)


# Riverbed only (no tension/adaptive_k, pure score fusion)
def riverbed_only(b, d, bw=0.8, dw=1.4):
    def norm(results):
        if not results:
            return {}
        vals = [s for _, s in results]
        mn, mx = min(vals), max(vals)
        rng = mx - mn if mx > mn else 1.0
        return {did: (s - mn) / rng for did, s in results}
    b_n, d_n = norm(b), norm(d)
    all_docs = set(b_n) | set(d_n)
    tw = bw + dw
    final = {
        did: (bw * b_n.get(did, 0) + dw * d_n.get(did, 0)) / tw
        for did in all_docs
    }
    return sorted(final.items(), key=lambda x: x[1], reverse=True)


# Tension only (adaptive RRF, no riverbed score)
def tension_only(b, d, k_low=3, k_high=10, top_n=20,
                 boost_max=1.2, bw=0.8, dw=1.4):
    b_set = {did for did, _ in b[:top_n]}
    d_set = {did for did, _ in d[:top_n]}
    union = b_set | d_set
    agreement = len(b_set & d_set) / len(union) if union else 0.0
    tension = 1.0 - agreement
    adaptive_k = max(1, int(k_low + (k_high - k_low) * tension))
    boost = 1.0 + (boost_max - 1.0) * agreement
    scores = defaultdict(float)
    presence = defaultdict(int)
    for rank, (did, _) in enumerate(b):
        scores[did] += bw / (adaptive_k + rank + 1)
        presence[did] += 1
    for rank, (did, _) in enumerate(d):
        scores[did] += dw / (adaptive_k + rank + 1)
        presence[did] += 1
    for did in scores:
        if presence[did] >= 2:
            scores[did] *= boost
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


print("All fusion functions + official evaluator ready")

All fusion functions + official evaluator ready


In [4]:
# ── Load SciFact ──

BASE_DIR = "datasets"
os.makedirs(BASE_DIR, exist_ok=True)

SCIFACT_URL = (
    "https://public.ukp.informatik.tu-darmstadt.de"
    "/thakur/BEIR/datasets/scifact.zip"
)
data_path = os.path.join(BASE_DIR, "scifact")
if not os.path.isdir(data_path):
    data_path = util.download_and_unzip(SCIFACT_URL, BASE_DIR)

corpus, queries, qrels = GenericDataLoader(data_path).load(split="test")
print(f"Corpus: {len(corpus)} | Queries: {len(queries)}")

datasets/scifact.zip:   0%|          | 0.00/2.69M [00:00<?, ?iB/s]

  0%|          | 0/5183 [00:00<?, ?it/s]

Corpus: 5183 | Queries: 300


In [5]:
# ── BM25 index ──

def tokenize(text):
    return re.findall(r'\w+', text.lower())

bm25_ids = list(corpus.keys())
tokenized = [
    tokenize(
        f"{corpus[did].get('title', '')} "
        f"{corpus[did].get('text', '')}"
    )
    for did in bm25_ids
]
bm25 = BM25Okapi(tokenized)

def search_bm25(query, top_k=100):
    scores = bm25.get_scores(tokenize(query))
    top_idx = scores.argsort()[-top_k:][::-1]
    return [
        (bm25_ids[i], float(scores[i]))
        for i in top_idx if scores[i] > 0
    ]

print(f"BM25 ready ({len(bm25_ids)} docs)")

BM25 ready (5183 docs)


In [6]:
# ── Dense index (E5-PT base) ──

model = SentenceTransformer(
    "intfloat/e5-base-unsupervised", device=DEVICE,
)

CACHE_PATH = os.path.join(BASE_DIR, ".cache_scifact_e5pt.npz")
doc_id_list = list(corpus.keys())

if os.path.exists(CACHE_PATH):
    passage_embs = np.load(CACHE_PATH)["embs"]
    print(f"Cache hit: {passage_embs.shape}")
else:
    texts = [
        f"passage: {corpus[did].get('title', '')} "
        f"{corpus[did].get('text', '')}".strip()
        for did in doc_id_list
    ]
    passage_embs = model.encode(
        texts, normalize_embeddings=True,
        show_progress_bar=True, batch_size=128,
    )
    np.savez_compressed(CACHE_PATH, embs=passage_embs)
    print(f"Encoded + saved: {passage_embs.shape}")

def dense_search(query, top_k=100):
    q = model.encode(
        ["query: " + query], normalize_embeddings=True,
    )
    sims = (passage_embs @ q.T).flatten()
    idx = np.argsort(sims)[::-1][:top_k]
    return [(doc_id_list[i], float(sims[i])) for i in idx]

print("Dense ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/700 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/41 [00:00<?, ?it/s]

Encoded + saved: (5183, 768)
Dense ready


In [7]:
# ── Load reranker ──

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    max_length=512,
    device=DEVICE,
)
print("Reranker loaded")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Reranker loaded


In [8]:
# ── Cache all query results ──

print("Caching queries...")
cached = {}
for qid, qt in queries.items():
    rel = {d: r for d, r in qrels.get(qid, {}).items() if r > 0}
    cached[qid] = {
        "query": qt,
        "bm25": search_bm25(qt, top_k=100),
        "dense": dense_search(qt, top_k=100),
        "rel": rel,
    }
print(f"Cached {len(cached)} queries")

Caching queries...
Cached 300 queries


In [9]:
# ── Reranker helper ──

def rerank_results(query_text, results, top_k=20):
    """Rerank top_k results with cross-encoder.
    Returns full list: reranked top + remaining tail.
    """
    if not results or top_k <= 0:
        return results
    top = results[:top_k]
    tail = results[top_k:]

    pairs = []
    valid_ids = []
    for did, _ in top:
        if did in corpus:
            doc = corpus[did]
            text = f"{doc.get('title', '')} {doc.get('text', '')}"
            pairs.append((query_text, text))
            valid_ids.append(did)

    if not pairs:
        return results

    ce_scores = reranker.predict(pairs)
    reranked = sorted(
        zip(valid_ids, ce_scores),
        key=lambda x: x[1], reverse=True,
    )
    reranked_set = set(valid_ids)
    clean_tail = [
        (did, s) for did, s in tail if did not in reranked_set
    ]
    return [(did, float(s)) for did, s in reranked] + clean_tail


print("Reranker helper ready")

Reranker helper ready


In [10]:
# ── Core evaluation ──

def eval_pipeline(pipeline_fn):
    """Evaluate a pipeline function on all queries using pytrec_eval.
    pipeline_fn(entry) -> [(doc_id, score), ...]
    Returns dict with ndcg_cut_10, recall_100, map.
    """
    run_dict = {}
    for qid, e in cached.items():
        fused = pipeline_fn(e)
        run_dict[qid] = {did: float(score) for did, score in fused[:100]}
    return evaluate_official(qrels, run_dict)


print("Starting ablation experiments...")
print("=" * 70)

results = {}

# ── Group 1: Baselines (no reranker) ──
print("\n--- Group 1: Baselines (no reranker) ---")

m = eval_pipeline(lambda e: e["dense"])
results["dense_only"] = m
print(f"Dense only:          {m['ndcg_cut_10']:.4f}")

m = eval_pipeline(lambda e: e["bm25"])
results["bm25_only"] = m
print(f"BM25 only:           {m['ndcg_cut_10']:.4f}")

m = eval_pipeline(
    lambda e: simple_rrf(e["bm25"], e["dense"]),
)
results["rrf"] = m
print(f"Simple RRF:          {m['ndcg_cut_10']:.4f}")

m = eval_pipeline(
    lambda e: riverbed_only(e["bm25"], e["dense"]),
)
results["riverbed_only"] = m
print(f"Riverbed only:       {m['ndcg_cut_10']:.4f}")

m = eval_pipeline(
    lambda e: tension_only(e["bm25"], e["dense"]),
)
results["tension_only"] = m
print(f"Tension only:        {m['ndcg_cut_10']:.4f}")

m = eval_pipeline(
    lambda e: riverbed_tension(e["bm25"], e["dense"]),
)
results["rt_full"] = m
print(f"RT full (baseline):  {m['ndcg_cut_10']:.4f}")

Starting ablation experiments...

--- Group 1: Baselines (no reranker) ---
Dense only:          0.7371
BM25 only:           0.6519
Simple RRF:          0.7503
Riverbed only:       0.7576
Tension only:        0.7485
RT full (baseline):  0.7557


In [11]:
# ── Group 2: Single-leg + reranker ──
print("\n--- Group 2: Single-leg + reranker ---")

for depth in [10, 20, 50]:
    print(f"\n  rerank depth = {depth}:")

    # Dense → reranker
    m = eval_pipeline(
        lambda e, d=depth: rerank_results(
            e["query"], e["dense"], top_k=d,
        ),
    )
    key = f"dense_rerank_{depth}"
    results[key] = m
    print(f"    Dense → rerank({depth}):  {m['ndcg_cut_10']:.4f}")

    # BM25 → reranker
    m = eval_pipeline(
        lambda e, d=depth: rerank_results(
            e["query"], e["bm25"], top_k=d,
        ),
    )
    key = f"bm25_rerank_{depth}"
    results[key] = m
    print(f"    BM25 → rerank({depth}):   {m['ndcg_cut_10']:.4f}")


--- Group 2: Single-leg + reranker ---

  rerank depth = 10:
    Dense → rerank(10):  0.5942
    BM25 → rerank(10):   0.0323

  rerank depth = 20:
    Dense → rerank(20):  0.5854
    BM25 → rerank(20):   0.0089

  rerank depth = 50:
    Dense → rerank(50):  0.5827
    BM25 → rerank(50):   0.0050


In [12]:
# ── Group 3: Fusion + reranker ──
print("\n--- Group 3: Fusion + reranker ---")

for depth in [10, 20, 50]:
    print(f"\n  rerank depth = {depth}:")

    # RRF → reranker
    m = eval_pipeline(
        lambda e, d=depth: rerank_results(
            e["query"],
            simple_rrf(e["bm25"], e["dense"]),
            top_k=d,
        ),
    )
    key = f"rrf_rerank_{depth}"
    results[key] = m
    print(f"    RRF → rerank({depth}):    {m['ndcg_cut_10']:.4f}")

    # RT → reranker
    m = eval_pipeline(
        lambda e, d=depth: rerank_results(
            e["query"],
            riverbed_tension(e["bm25"], e["dense"]),
            top_k=d,
        ),
    )
    key = f"rt_rerank_{depth}"
    results[key] = m
    print(f"    RT → rerank({depth}):     {m['ndcg_cut_10']:.4f}")


--- Group 3: Fusion + reranker ---

  rerank depth = 10:
    RRF → rerank(10):    0.6146
    RT → rerank(10):     0.6088

  rerank depth = 20:
    RRF → rerank(20):    0.6000
    RT → rerank(20):     0.5947

  rerank depth = 50:
    RRF → rerank(50):    0.5987
    RT → rerank(50):     0.5988


In [13]:
# ── Group 4: Reranker FIRST, then fusion (key insight!) ──
print("\n--- Group 4: Reranker first → then RT fusion ---")

for depth in [10, 20, 50]:
    print(f"\n  rerank depth = {depth}:")

    # Dense → reranker → RT with raw BM25
    m = eval_pipeline(
        lambda e, d=depth: riverbed_tension(
            e["bm25"],  # BM25 stays raw
            rerank_results(e["query"], e["dense"], top_k=d),
        ),
    )
    key = f"dense_rerank_then_rt_{depth}"
    results[key] = m
    print(f"    Dense→rerank({depth})→RT:  {m['ndcg_cut_10']:.4f}")

    # Both legs reranked → RT
    m = eval_pipeline(
        lambda e, d=depth: riverbed_tension(
            rerank_results(e["query"], e["bm25"], top_k=d),
            rerank_results(e["query"], e["dense"], top_k=d),
        ),
    )
    key = f"both_rerank_then_rt_{depth}"
    results[key] = m
    print(f"    Both→rerank({depth})→RT:   {m['ndcg_cut_10']:.4f}")

    # Dense → reranker → RRF with raw BM25
    m = eval_pipeline(
        lambda e, d=depth: simple_rrf(
            e["bm25"],
            rerank_results(e["query"], e["dense"], top_k=d),
        ),
    )
    key = f"dense_rerank_then_rrf_{depth}"
    results[key] = m
    print(f"    Dense→rerank({depth})→RRF: {m['ndcg_cut_10']:.4f}")


--- Group 4: Reranker first → then RT fusion ---

  rerank depth = 10:
    Dense→rerank(10)→RT:  0.7222
    Both→rerank(10)→RT:   0.7040
    Dense→rerank(10)→RRF: 0.7316

  rerank depth = 20:
    Dense→rerank(20)→RT:  0.7140
    Both→rerank(20)→RT:   0.6942
    Dense→rerank(20)→RRF: 0.7166

  rerank depth = 50:
    Dense→rerank(50)→RT:  0.7120
    Both→rerank(50)→RT:   0.6967
    Dense→rerank(50)→RRF: 0.7065


In [14]:
# ── Group 5: Component ablation for RT ──
print("\n--- Group 5: RT component ablation + reranker ---")

for depth in [10, 20]:
    print(f"\n  rerank depth = {depth}:")

    # Riverbed only → reranker
    m = eval_pipeline(
        lambda e, d=depth: rerank_results(
            e["query"],
            riverbed_only(e["bm25"], e["dense"]),
            top_k=d,
        ),
    )
    key = f"riverbed_rerank_{depth}"
    results[key] = m
    print(f"    Riverbed→rerank({depth}):  {m['ndcg_cut_10']:.4f}")

    # Tension only → reranker
    m = eval_pipeline(
        lambda e, d=depth: rerank_results(
            e["query"],
            tension_only(e["bm25"], e["dense"]),
            top_k=d,
        ),
    )
    key = f"tension_rerank_{depth}"
    results[key] = m
    print(f"    Tension→rerank({depth}):   {m['ndcg_cut_10']:.4f}")


print("\n" + "=" * 70)
print("ALL EXPERIMENTS DONE")
print("=" * 70)


--- Group 5: RT component ablation + reranker ---

  rerank depth = 10:
    Riverbed→rerank(10):  0.6132
    Tension→rerank(10):   0.6164

  rerank depth = 20:
    Riverbed→rerank(20):  0.5955
    Tension→rerank(20):   0.6020

ALL EXPERIMENTS DONE


In [15]:
# ── Summary ──

print("\n" + "=" * 70)
print("RERANKER ABLATION SUMMARY — SciFact nDCG@10 (pytrec_eval official)")
print("=" * 70)

# Sort by ndcg_cut_10 descending
sorted_results = sorted(
    results.items(),
    key=lambda x: x[1]["ndcg_cut_10"],
    reverse=True,
)

rt_base = results.get("rt_full", {}).get("ndcg_cut_10", 0)
print(f"\n{'Rank':<5} {'Pipeline':<35} {'nDCG@10':>8} {'R@100':>8} {'MAP':>8} {'vs RT':>8}")
print("-" * 78)
for i, (name, scores) in enumerate(sorted_results):
    n = scores["ndcg_cut_10"]
    r = scores.get("recall_100", 0)
    m = scores.get("map", 0)
    delta = n - rt_base
    marker = " ★" if n >= rt_base else ""
    print(
        f"{i+1:<5} {name:<35} {n:>8.4f} {r:>8.4f} {m:>8.4f} "
        f"{delta:>+8.4f}{marker}"
    )

print(f"\nRT baseline: {rt_base:.4f}")
print(f"Best overall: {sorted_results[0][0]} = {sorted_results[0][1]['ndcg_cut_10']:.4f}")

# Key question answers
print("\n" + "=" * 70)
print("KEY FINDINGS")
print("=" * 70)

# Does reranker help on simple input?
dense_base = results.get("dense_only", {}).get("ndcg_cut_10", 0)
dense_rr10 = results.get("dense_rerank_10", {}).get("ndcg_cut_10", 0)
dense_rr20 = results.get("dense_rerank_20", {}).get("ndcg_cut_10", 0)
print(f"\n1. Does reranker help Dense alone?")
print(f"   Dense: {dense_base:.4f} → +rerank(10): {dense_rr10:.4f} → +rerank(20): {dense_rr20:.4f}")

# Does reranker-first then RT beat RT alone?
dr_rt_20 = results.get("dense_rerank_then_rt_20", {}).get("ndcg_cut_10", 0)
print(f"\n2. Does reranker-first → RT beat RT alone?")
print(f"   RT: {rt_base:.4f} vs Dense→rerank(20)→RT: {dr_rt_20:.4f} (Δ={dr_rt_20-rt_base:+.4f})")

# Does RT → reranker hurt?
rt_rr10 = results.get("rt_rerank_10", {}).get("ndcg_cut_10", 0)
rt_rr20 = results.get("rt_rerank_20", {}).get("ndcg_cut_10", 0)
print(f"\n3. Does RT → reranker still hurt?")
print(f"   RT: {rt_base:.4f} → +rerank(10): {rt_rr10:.4f} → +rerank(20): {rt_rr20:.4f}")

# Best pipeline overall
print(f"\n4. Best pipeline: {sorted_results[0][0]} = {sorted_results[0][1]['ndcg_cut_10']:.4f}")


RERANKER ABLATION SUMMARY — SciFact nDCG@10 (pytrec_eval official)

Rank  Pipeline                             nDCG@10    R@100      MAP    vs RT
------------------------------------------------------------------------------
1     riverbed_only                         0.7576   0.9767   0.7175  +0.0019 ★
2     rt_full                               0.7557   0.9767   0.7141  +0.0000 ★
3     rrf                                   0.7503   0.9767   0.7057  -0.0054
4     tension_only                          0.7485   0.9767   0.7053  -0.0072
5     dense_only                            0.7371   0.9800   0.6923  -0.0186
6     dense_rerank_then_rrf_10              0.7316   0.9767   0.6803  -0.0241
7     dense_rerank_then_rt_10               0.7222   0.9843   0.6832  -0.0335
8     dense_rerank_then_rrf_20              0.7166   0.9767   0.6668  -0.0391
9     dense_rerank_then_rt_20               0.7140   0.9810   0.6721  -0.0416
10    dense_rerank_then_rt_50               0.7120   0.9760   0.6672

In [16]:
# ── Save results ──

output = {
    "experiment": "reranker_ablation",
    "dataset": "scifact",
    "embedding_model": "intfloat/e5-base-unsupervised",
    "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "device": DEVICE,
    "evaluator": "pytrec_eval 0.5.10 (official BEIR standard)",
    "submission_ready": True,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "results": results,
    "sorted_ranking": [
        {
            "pipeline": name,
            "ndcg_cut_10": scores["ndcg_cut_10"],
            "recall_100": scores.get("recall_100", 0),
            "map": scores.get("map", 0),
        }
        for name, scores in sorted_results
    ],
}

out_file = "reranker_ablation.json"
with open(out_file, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved: {out_file}")
print("\n=== COPY THIS JSON ===")
print(json.dumps(output, indent=2))
print("=== END JSON ===")

Saved: reranker_ablation.json

=== COPY THIS JSON ===
{
  "experiment": "reranker_ablation",
  "dataset": "scifact",
  "embedding_model": "intfloat/e5-base-unsupervised",
  "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
  "device": "cuda",
  "evaluator": "pytrec_eval 0.5.10 (official BEIR standard)",
  "submission_ready": true,
  "timestamp": "2026-03-28 01:40:26",
  "results": {
    "dense_only": {
      "recall_100": 0.98,
      "ndcg_cut_10": 0.737066,
      "map": 0.692277
    },
    "bm25_only": {
      "recall_100": 0.873056,
      "ndcg_cut_10": 0.651893,
      "map": 0.613182
    },
    "rrf": {
      "recall_100": 0.976667,
      "ndcg_cut_10": 0.750332,
      "map": 0.705718
    },
    "riverbed_only": {
      "recall_100": 0.976667,
      "ndcg_cut_10": 0.757596,
      "map": 0.717546
    },
    "tension_only": {
      "recall_100": 0.976667,
      "ndcg_cut_10": 0.74846,
      "map": 0.705312
    },
    "rt_full": {
      "recall_100": 0.976667,
      "ndcg_cut_1